# Use Qdrant In NAIVE RAG

In [1]:
# Import dataset and libraries
import warnings
warnings.filterwarnings('ignore')

from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader('Why_Language_Models_Hallucinate_Explainer.pdf')
pages = loader.load()
print(f'Total pages loaded: {len(pages)}')


Total pages loaded: 3


In [2]:
# Split Dataset into Chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=150)
texts = text_splitter.split_documents(pages)
chunks = [i.page_content for i in texts]
metadata = [i.metadata for i in texts]
print(f'Total chunks created: {len(chunks)}')
metadata[0]


Total chunks created: 10


{'producer': 'ReportLab PDF Library - (opensource)',
 'creator': '(unspecified)',
 'creationdate': '2026-07-02T09:06:07+00:00',
 'author': '(anonymous)',
 'keywords': '',
 'moddate': '2026-07-02T09:06:07+00:00',
 'subject': '(unspecified)',
 'title': '(anonymous)',
 'trapped': '/False',
 'source': 'Why_Language_Models_Hallucinate_Explainer.pdf',
 'total_pages': 3,
 'page': 0,
 'page_label': '1'}

In [3]:
# Create Embeddings using Sentence Transformers
from sentence_transformers import SentenceTransformer
embed_transformer = SentenceTransformer(model_name_or_path='all-MiniLM-L6-v2', similarity_fn_name='cosine')
embeddings = embed_transformer.encode(chunks)
print(f'Embedding dimension: {len(embeddings[0])}')


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6662.25it/s]


Embedding dimension: 384


In [4]:
# Initialize Qdrant Client
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

load_dotenv()

collection_name = 'Model_Halucination'

# Safely close any existing local client connection to prevent file locking issues
if 'client' in globals() and client is not None:
    try:
        client.close()
    except Exception:
        pass

# Connect using local disk storage
client = QdrantClient(path='./Naive_RAG')



In [5]:
# Create Collection if it does not exist
if not client.collection_exists(collection_name=collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE)
    )
    print(f"Collection '{collection_name}' created successfully.")
else:
    print(f"Collection '{collection_name}' already exists.")


Collection 'Model_Halucination' already exists.


In [6]:
# Prepare and Upsert Vectors to Qdrant
points = []
for i, (chunk, meta) in enumerate(zip(chunks, metadata)):
    vector = embed_transformer.encode(chunk).tolist()
    payload = dict(meta)
    payload['text'] = chunk
    points.append(PointStruct(id=i, vector=vector, payload=payload))

client.upsert(collection_name=collection_name, points=points)
print(f"Successfully upserted {len(points)} points into collection '{collection_name}'.")


Successfully upserted 10 points into collection 'Model_Halucination'.


In [7]:
# Retrieval Function
def retrieve_chunks(query: str, threshold: float = 0.3, k_top: int = 3):
    query_encode = embed_transformer.encode(query).tolist()
    result = client.query_points(collection_name=collection_name, query=query_encode, limit=k_top).points
    near_chunks = [r.payload['text'] for r in result if r.score is None or r.score >= threshold]
    return '\n\n'.join(near_chunks) if near_chunks else 'Not Relevant Content'


In [9]:
# Query Qdrant and Generate Answer with Groq LLM
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

q1 = 'why we need to learn LLM ?'
content = retrieve_chunks(query=q1)

prompt = f"""
Provide answers to the user questions based on the provided data:
Content: {content}
Question: {q1}
"""

groq = ChatGroq(model='llama-3.1-8b-instant', api_key=os.getenv('GROQ_API_KEY'))
response = groq.invoke(prompt).content
print(response)


Based on the provided data, the answer to the question "why we need to learn LLM" is not explicitly mentioned in the provided content. However, it can be inferred that the purpose of learning LLM is likely for natural language processing, understanding language, and generating human-like text responses.

From the content, it is clear that Large Language Models (LLMs) are being trained and studied for tasks such as text generation, classification, and evaluation. The goal of learning LLM is likely to improve its ability to understand and generate human language, making it a useful tool for various applications, such as chatbots, language translation, and text summarization.

In other words, learning LLM is necessary to develop more accurate and informative language models that can assist humans in various tasks and applications.
